Step 1 — Setup

In [ ]:
import os, json, random
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from transformers import BertModel, BertTokenizer
from torch_geometric.nn import SAGEConv, global_mean_pool
from torch_geometric.data import Batch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from sklearn.metrics import f1_score

PROJECT_ROOT = os.getcwd()
RAW = os.path.join(PROJECT_ROOT, "data/raw")
PROC = os.path.join(PROJECT_ROOT, "data/processed")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
MC_GRAPH_DIR = os.path.join(PROC, "musiccaps_graphs")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


import ast
mc = pd.read_csv(os.path.join(RAW, "musiccaps/musiccaps-public.csv"))
mc_graph_manifest = pd.read_csv(os.path.join(PROC, "musiccaps_graph_manifest.csv"))
successful_ids = set(mc_graph_manifest[mc_graph_manifest["success"] == True]["ytid"])
mc_final = mc[mc["ytid"].isin(successful_ids)].reset_index(drop=True)
mc_final["aspect_parsed"] = mc_final["aspect_list"].apply(ast.literal_eval)

from collections import Counter
all_aspects = [a for aspects in mc_final["aspect_parsed"] for a in aspects]
top30_aspects = [phrase for phrase, count in Counter(all_aspects).most_common(30)]
for tag in top30_aspects:
    mc_final[tag] = mc_final["aspect_parsed"].apply(lambda aspects: 1 if tag in aspects else 0)

random.seed(42)  
indices = list(range(len(mc_final)))
random.shuffle(indices)
n = len(indices)
train_end, val_end = int(0.8*n), int(0.9*n)
mc_train = mc_final.iloc[indices[:train_end]].reset_index(drop=True)
mc_val = mc_final.iloc[indices[train_end:val_end]].reset_index(drop=True)
mc_test = mc_final.iloc[indices[val_end:]].reset_index(drop=True)
print("Train:", len(mc_train), "| Val:", len(mc_val), "| Test:", len(mc_test))

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

d:\Program Files\Python\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train: 927 | Val: 116 | Test: 116


Step 2 — Dual encoder model + dataset

In [ ]:
class GraphEncoder(nn.Module):
    def __init__(self, hidden=64, embed_dim=128):
        super().__init__()
        self.conv1 = SAGEConv(12, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.proj = nn.Linear(hidden, embed_dim)
    def forward(self, graph_batch):
        x = F.relu(self.conv1(graph_batch.x, graph_batch.edge_index))
        x = F.relu(self.conv2(x, graph_batch.edge_index))
        g = global_mean_pool(x, graph_batch.batch)
        return F.normalize(self.proj(g), dim=-1)  

class TextEncoder(nn.Module):
    def __init__(self, embed_dim=128):
        super().__init__()
        self.bert = BertModel.from_pretrained("bert-base-uncased")
        self.proj = nn.Linear(768, embed_dim)
    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        return F.normalize(self.proj(cls), dim=-1)

class ContrastiveDataset(Dataset):
    def __init__(self, df, graph_dir, tokenizer, max_length=128):
        self.df = df.reset_index(drop=True)
        self.graph_dir = graph_dir
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        graph = torch.load(os.path.join(self.graph_dir, f"{row['ytid']}.pt"), weights_only=False)
        tokens = self.tokenizer(row["caption"], padding="max_length", truncation=True,
                                 max_length=self.max_length, return_tensors="pt")
        return graph, tokens["input_ids"].squeeze(0), tokens["attention_mask"].squeeze(0)

def contrastive_collate(batch):
    graphs, input_ids, attn_masks = zip(*batch)
    return Batch.from_data_list(list(graphs)), torch.stack(input_ids), torch.stack(attn_masks)

BATCH_SIZE_CONTRASTIVE = 16  

train_loader_c = DataLoader(ContrastiveDataset(mc_train, MC_GRAPH_DIR, tokenizer), batch_size=BATCH_SIZE_CONTRASTIVE,
                             shuffle=True, collate_fn=contrastive_collate, drop_last=True)  
val_loader_c = DataLoader(ContrastiveDataset(mc_val, MC_GRAPH_DIR, tokenizer), batch_size=BATCH_SIZE_CONTRASTIVE,
                           collate_fn=contrastive_collate)
test_loader_c = DataLoader(ContrastiveDataset(mc_test, MC_GRAPH_DIR, tokenizer), batch_size=BATCH_SIZE_CONTRASTIVE,
                            collate_fn=contrastive_collate)

graph_encoder = GraphEncoder().to(device)
text_encoder = TextEncoder().to(device)
print("Models created.")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2694.61it/s]
[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Models created.


Step 3 — InfoNCE loss + training loop

In [ ]:
def info_nce_loss(graph_embeds, text_embeds, temperature=0.07):
    # Similarity matrix: every graph vs every caption in the batch
    logits = torch.matmul(graph_embeds, text_embeds.T) / temperature  
    labels = torch.arange(logits.size(0)).to(device)  
    loss_g2t = F.cross_entropy(logits, labels)          
    loss_t2g = F.cross_entropy(logits.T, labels)         
    return (loss_g2t + loss_t2g) / 2

optimizer_c = AdamW(list(graph_encoder.parameters()) + list(text_encoder.parameters()), lr=2e-5)

contrastive_history = {"train_loss": [], "val_loss": []}
best_val_loss = float("inf")
best_states = None
patience, epochs_no_improve = 3, 0

for epoch in range(15):
    graph_encoder.train(); text_encoder.train()
    train_loss = 0
    for graph_batch, input_ids, attn_mask in tqdm(train_loader_c, desc=f"Epoch {epoch+1}"):
        graph_batch, input_ids, attn_mask = graph_batch.to(device), input_ids.to(device), attn_mask.to(device)
        optimizer_c.zero_grad()
        g_emb = graph_encoder(graph_batch)
        t_emb = text_encoder(input_ids, attn_mask)
        loss = info_nce_loss(g_emb, t_emb)
        loss.backward()
        optimizer_c.step()
        train_loss += loss.item()

    graph_encoder.eval(); text_encoder.eval()
    val_loss = 0
    with torch.no_grad():
        for graph_batch, input_ids, attn_mask in val_loader_c:
            graph_batch, input_ids, attn_mask = graph_batch.to(device), input_ids.to(device), attn_mask.to(device)
            g_emb = graph_encoder(graph_batch)
            t_emb = text_encoder(input_ids, attn_mask)
            if g_emb.size(0) > 1:  
                val_loss += info_nce_loss(g_emb, t_emb).item()
    val_loss /= len(val_loader_c)

    contrastive_history["train_loss"].append(train_loss/len(train_loader_c))
    contrastive_history["val_loss"].append(val_loss)
    print(f"Epoch {epoch+1}: train_loss={train_loss/len(train_loader_c):.4f} | val_loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_states = {"graph": {k: v.cpu().clone() for k, v in graph_encoder.state_dict().items()},
                        "text": {k: v.cpu().clone() for k, v in text_encoder.state_dict().items()}}
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

graph_encoder.load_state_dict(best_states["graph"])
text_encoder.load_state_dict(best_states["text"])

Epoch 1: 100%|██████████| 57/57 [00:28<00:00,  2.01it/s]


Epoch 1: train_loss=2.7644 | val_loss=2.4357


Epoch 2: 100%|██████████| 57/57 [00:23<00:00,  2.42it/s]


Epoch 2: train_loss=2.5930 | val_loss=2.2997


Epoch 3: 100%|██████████| 57/57 [00:23<00:00,  2.38it/s]


...

Epoch 6: 100%|██████████| 57/57 [00:28<00:00,  1.99it/s]


Epoch 6: train_loss=2.0314 | val_loss=2.3606


Epoch 7: 100%|██████████| 57/57 [00:27<00:00,  2.11it/s]


Epoch 7: train_loss=1.8260 | val_loss=2.4655
Early stopping at epoch 7


<All keys matched successfully>

Step 4 — Retrieval evaluation (R@1, R@5, R@10)

In [ ]:
def get_all_embeddings(loader):
    graph_encoder.eval(); text_encoder.eval()
    all_g, all_t = [], []
    with torch.no_grad():
        for graph_batch, input_ids, attn_mask in loader:
            graph_batch, input_ids, attn_mask = graph_batch.to(device), input_ids.to(device), attn_mask.to(device)
            all_g.append(graph_encoder(graph_batch).cpu())
            all_t.append(text_encoder(input_ids, attn_mask).cpu())
    return torch.cat(all_g), torch.cat(all_t)

test_g_embeds, test_t_embeds = get_all_embeddings(test_loader_c)
print("Test embeddings:", test_g_embeds.shape, test_t_embeds.shape)

def compute_recall_at_k(query_embeds, gallery_embeds, k_values=[1, 5, 10]):
    sim_matrix = torch.matmul(query_embeds, gallery_embeds.T)  
    n = sim_matrix.size(0)
    correct_idx = torch.arange(n)  

    results = {}
    for k in k_values:
        topk = sim_matrix.topk(k, dim=1).indices  # (n, k)
        hits = (topk == correct_idx.unsqueeze(1)).any(dim=1)
        results[f"R@{k}"] = hits.float().mean().item()
    return results

print("Caption -> Audio retrieval (given a caption, find the matching graph):")
c2a = compute_recall_at_k(test_t_embeds, test_g_embeds)
print(c2a)

print("Audio -> Caption retrieval (given a graph, find the matching caption):")
a2c = compute_recall_at_k(test_g_embeds, test_t_embeds)
print(a2c)

Test embeddings: torch.Size([116, 128]) torch.Size([116, 128])
Caption -> Audio retrieval (given a caption, find the matching graph):
{'R@1': 0.017241379246115685, 'R@5': 0.09482758492231369, 'R@10': 0.2068965584039688}
Audio -> Caption retrieval (given a graph, find the matching caption):
{'R@1': 0.017241379246115685, 'R@5': 0.08620689809322357, 'R@10': 0.1551724076271057}


Step 5 — 10 qualitative retrieval examples

In [5]:
qualitative_examples = []
for i in range(10):
    query_caption = mc_test.iloc[i]["caption"]
    sim_scores = torch.matmul(test_t_embeds[i], test_g_embeds.T)
    top3_idx = sim_scores.topk(3).indices.tolist()

    matched_ytids = [mc_test.iloc[j]["ytid"] for j in top3_idx]
    is_correct_in_top3 = i in top3_idx

    qualitative_examples.append({
        "query_caption": query_caption,
        "true_ytid": mc_test.iloc[i]["ytid"],
        "top3_matched_ytids": matched_ytids,
        "correct_match_in_top3": is_correct_in_top3
    })
    print(f"Query: {query_caption[:100]}...")
    print(f"True match: {mc_test.iloc[i]['ytid']} | Top-3 retrieved: {matched_ytids} | Correct in top-3: {is_correct_in_top3}")
    print("-" * 80)

with open(os.path.join(RESULTS_DIR, "retrieval_examples/task4_qualitative_examples.json"), "w") as f:
    json.dump(qualitative_examples, f, indent=2)

Query: The low quality recording features a tutorial that contains a flat male vocal talking over acoustic ...
True match: 8UhdwnsckJ8 | Top-3 retrieved: ['B3lq6U4PDZo', '4Mo7tdV2LZk', '-Vo4CAMX26U'] | Correct in top-3: False
--------------------------------------------------------------------------------
Query: This jazz song features a saxophone playing the main melody. This is accompanied by percussion playi...
True match: 50fuQm8B2Yg | Top-3 retrieved: ['9K8EePrEDdo', '7S3fU4RHabw', '7Vjp9y6wvkY'] | Correct in top-3: False
--------------------------------------------------------------------------------
Query: This instrumental song features a melody played using bells. There are chiming sounds played in the ...
True match: B3lq6U4PDZo | Top-3 retrieved: ['4ueN2gGsH5Y', '9K8EePrEDdo', '7S3fU4RHabw'] | Correct in top-3: False
--------------------------------------------------------------------------------
Query: This audio clip is a harmonica melody with vocalisation.The music is dis

Step 6 — Zero-shot tag prediction from captions vs. Task 3's supervised model

In [ ]:
with open(os.path.join(RESULTS_DIR, "task3_fusion_history.json")) as f:
    task3_history = json.load(f)


with open(os.path.join(RESULTS_DIR, "task3_ablation_results.json")) as f:
    task3_ablation_results = json.load(f)

test_macro_f1_fusion = task3_ablation_results["Cross-Attention (Fusion)"]["test_macro_f1"]
test_micro_f1_fusion = task3_ablation_results["Cross-Attention (Fusion)"]["test_micro_f1"]
print("Task 3 fusion test scores reloaded:", test_macro_f1_fusion, test_micro_f1_fusion)

Task 3 fusion test scores reloaded: 0.5328676757900734 0.7006651884700665


In [ ]:
tag_texts = [f"this song is {tag}" for tag in top30_aspects]
tag_tokens = tokenizer(tag_texts, padding="max_length", truncation=True, max_length=128, return_tensors="pt")

text_encoder.eval()
with torch.no_grad():
    tag_embeds = text_encoder(tag_tokens["input_ids"].to(device), tag_tokens["attention_mask"].to(device)).cpu()

# For each test clip's audio graph embedding, find which tags are most similar
zero_shot_preds = []
sim_to_tags = torch.matmul(test_g_embeds, tag_embeds.T)  
threshold = sim_to_tags.mean() + sim_to_tags.std()  

zero_shot_binary = (sim_to_tags > threshold).float().numpy()
true_labels_test = mc_test[top30_aspects].values

zero_shot_macro_f1 = f1_score(true_labels_test, zero_shot_binary, average="macro", zero_division=0)
zero_shot_micro_f1 = f1_score(true_labels_test, zero_shot_binary, average="micro", zero_division=0)

print(f"Zero-shot (contrastive): macro_f1={zero_shot_macro_f1:.4f} | micro_f1={zero_shot_micro_f1:.4f}")
print(f"Task 3 supervised (fusion): macro_f1={test_macro_f1_fusion:.4f} | micro_f1={test_micro_f1_fusion:.4f}")

Zero-shot (contrastive): macro_f1=0.0791 | micro_f1=0.1215
Task 3 supervised (fusion): macro_f1=0.5329 | micro_f1=0.7007


In [9]:
torch.save(graph_encoder.state_dict(), os.path.join(RESULTS_DIR, "task4_graph_encoder.pt"))
torch.save(text_encoder.state_dict(), os.path.join(RESULTS_DIR, "task4_text_encoder.pt"))

with open(os.path.join(RESULTS_DIR, "task4_history.json"), "w") as f:
    json.dump(contrastive_history, f, indent=2)

task4_results = {
    "retrieval_caption_to_audio": c2a,
    "retrieval_audio_to_caption": a2c,
    "zero_shot_tagging": {"macro_f1": zero_shot_macro_f1, "micro_f1": zero_shot_micro_f1},
    "task3_supervised_comparison": {"macro_f1": test_macro_f1_fusion, "micro_f1": test_micro_f1_fusion}
}
with open(os.path.join(RESULTS_DIR, "task4_retrieval_results.json"), "w") as f:
    json.dump(task4_results, f, indent=2)

print("Task 4 fully saved.")

Task 4 fully saved.
